# Biblical RAG LLM MVP 1: Document Retrieval Optimization

This notebook tests mistral:instruct's ability to read a Bible that is recursively chunked.

In [1]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import json
import os
from uuid import uuid4

# Open King James Bible.
data_dir = os.path.join(os.path.dirname(os.getcwd()), 'data')
kjv_df = pd.read_csv(os.path.join(data_dir, 'en_kjv.csv'), index_col='index')
kjv_df.head()

,language,translation,book,chapter,verse,text
index,,,,,,
0,en,kjv,Gen,1,1,In the beginning God created the heaven and th...
1,en,kjv,Gen,1,2,"And the earth was without form, and void; and ..."
2,en,kjv,Gen,1,3,"And God said, Let there be light: and there wa..."
3,en,kjv,Gen,1,4,"And God saw the light, that it was good: and G..."
4,en,kjv,Gen,1,5,"And God called the light Day, and the darkness..."


In [2]:
# Open metadata table.
book_metadata = pd.read_csv(os.path.join(data_dir, 'metadata', 'books.csv'))
book_metadata.loc[book_metadata['book'].isin(['Numbers', 'Jonah', 'Ruth', 'Mark', 'Titus', 'Revelation'])]

,book,chapters,verses,avg verse per chapter,testament,category,author,abbreviation
3,Numbers,36,1288,36,old,law,Moses,Num
7,Ruth,4,85,21,old,writing,Samuel,Ruth
31,Jonah,4,48,12,old,prophet,Jonah,Jonah
40,Mark,16,678,42,new,gospel,Mark,Mark
55,Titus,3,46,15,new,epistle,Paul,Titus
65,Revelation,22,404,18,new,revelation,John,Rev


In [3]:
psalm_metadata = pd.read_csv(os.path.join(data_dir, 'metadata', 'psalms.csv'))
psalm_metadata.loc[psalm_metadata['psalm'].isin([1, 5, 42, 50, 89, 90, 127])]

,psalm,author
0,1,Anonymous
4,5,David
41,42,Sons of Korah
49,50,Asaph
88,89,Ethan
89,90,Moses
126,127,Solomon


## Load the Vectorized Bible Data

In [6]:
from langchain_ollama import OllamaEmbeddings, OllamaLLM
from langchain_community.vectorstores import FAISS

llm = OllamaLLM(model='mistral:instruct')
embedding = OllamaEmbeddings(model='mistral:instruct')
vector_file = os.path.join(data_dir, 'bible_vectostore_chapters')
vectorstore = FAISS.load_local(vector_file, embedding, allow_dangerous_deserialization=True)#.as_retriever() if you want to load it as a retriever
print("File loaded.")

File loaded.


## Develop the RAG Chain
The current state of the RAG Chain is unsatisfactory. It provides the wrong texts oftentimes. There are some solutions to this problem which involve altering the user's prompt (a.k.a. query translation). This includes multi-query, RAG fusion, decomposition, step-back, and HyDE.

### HyDE

In [7]:
from langchain.chains import RetrievalQA

print("Begin instantiating retriever.")
retriever = vectorstore.as_retriever(
    #fetch_k=len(split_docs)//10,
    #k=len(split_docs)//100,
    search_type='similarity_score_threshold',
    search_kwargs={'score_threshold': 0.2}
)
print("Retriever instantiated.")
qa = RetrievalQA.from_chain_type(
    llm=llm,
    retriever=retriever
)
print("RAG chain instantiated.")

Begin instantiating retriever.
Retriever instantiated.
RAG chain instantiated.


In [14]:
from langchain.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

template = """You are a Bible expert. The answer to the following question is in the Bible:
Question: {question}
Passage:
"""
prompt_hyde = ChatPromptTemplate.from_template(template)
parser = StrOutputParser()
generate_docs_for_retrieval = (prompt_hyde | llm | parser)

question = "How long did the sun stay still in the Sky during Joshua's conquest?"
generate_docs_for_retrieval.invoke({'question':question})

' The event you are referring to can be found in Joshua 10:12-14. According to this passage, the Lord caused the sun to stop in the middle of the sky for about a full day (approximately 24 hours) while Israel defeated their enemies, the Amorites. This was not a normal solar day but rather a miraculous event as described in the Bible.'

In [15]:
# retrieve docs with previous chain
retrieval_chain = generate_docs_for_retrieval | retriever
retrieved_docs = retrieval_chain.invoke({'question': question})
retrieved_docs

[Document(metadata={'title': 'Genesis', 'citation': 'Genesis 12', 'author': 'Moses', 'book_index': 11}, page_content="Now the LORD had said unto Abram, Get thee out of thy country, and from thy kindred, and from thy father's house, unto a land that I will shew thee: And I will make of thee a great nation, and I will bless thee, and make thy name great; and thou shalt be a blessing: And I will bless them that bless thee, and curse him that curseth thee: and in thee shall all families of the earth be blessed. So Abram departed, as the LORD had spoken unto him; and Lot went with him: and Abram was seventy and five years old when he departed out of Haran. And Abram took Sarai his wife, and Lot his brother's son, and all their substance that they had gathered, and the souls that they had gotten in Haran; and they went forth to go into the land of Canaan; and into the land of Canaan they came. And Abram passed through the land unto the place of Sichem, unto the plain of Moreh. And the Canaan

In [16]:
# RAG
template = """Answer the following question based on this context:

{context}

Question: {question}
"""
prompt = ChatPromptTemplate.from_template(template)
final_rag_chain = (prompt | llm | parser)
final_rag_chain.invoke({'context': retrieved_docs, 'question': question})

" The question does not seem to be directly related to the provided text about Sodom and Gomorrah. In the Bible, it is mentioned that during Joshua's conquest of Jericho (Joshua 10:12-14), the sun stood still for a full day so that Israel could complete the battle. This event is not specifically mentioned in the text about Sodom and Gomorrah."